# EDA: regime detection (single-condition vs. 6-condition)

Exploration only, nothing here is load-bearing (see CLAUDE.md). Supports decisions made in `src/regimes.py`.

**Question 1:** does k=6 actually hold up in the raw operational-setting data for FD002/FD004, or was that just trusting the doc?

**Question 2:** `regimes.py` needs to auto-detect whether a dataset is single-condition (FD001/FD003, no-op) or 6-condition (FD002/FD004, k-means) using a std threshold on op1/op2/op3. What's a safe, well-justified threshold, and how much margin does it actually have?

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.load import load_train

%matplotlib inline

## Question 1: is k=6 real?

Round FD002's op-settings and count how many distinct combinations actually show up, with how many rows each.

In [2]:
df2 = load_train("FD002")
rounded = df2[["op1", "op2", "op3"]].round(1)
print(rounded.value_counts())

op1   op2  op3  
42.0  0.8  100.0    13458
20.0  0.7  100.0     8122
0.0   0.0  100.0     8044
35.0  0.8  100.0     8037
25.0  0.6  60.0      8002
10.0  0.2  100.0     4138
      0.3  100.0     3958
Name: count, dtype: int64


**Finding:** 7 rounded groups show up, but two of them (`op1=10.0, op2=0.2` and `op1=10.0, op2=0.3`) are the same real regime split by floating-point noise landing right on a rounding boundary (true value near op2=0.25). The real count is 6, confirmed. k=6 is not a blind trust of the doc.

## Question 2: how safe is the single-condition/multi-condition threshold?

`_is_single_condition()` in `src/regimes.py` checks whether op1/op2/op3's standard deviation is below a threshold, for all three columns. Compare the worst case on each side: the noisiest single-condition column vs. the tightest multi-condition column.

In [3]:
import pandas as pd

rows = []
for name in ["FD001", "FD002", "FD003", "FD004"]:
    d = load_train(name)
    rows.append({"dataset": name, **d[["op1", "op2", "op3"]].std().to_dict()})

std_table = pd.DataFrame(rows).set_index("dataset")
print(std_table)

single_cols = std_table.loc[["FD001", "FD003"]]
multi_cols = std_table.loc[["FD002", "FD004"]]
print()
print("Single-condition worst (max) per column:")
print(single_cols.max())
print()
print("Multi-condition tightest (min) per column:")
print(multi_cols.min())

               op1       op2        op3
dataset                                
FD001     0.002187  0.000293   0.000000
FD002    14.747376  0.310016  14.237735
FD003     0.002194  0.000294   0.000000
FD004    14.780722  0.310703  14.251954

Single-condition worst (max) per column:
op1    0.002194
op2    0.000294
op3    0.000000
dtype: float64

Multi-condition tightest (min) per column:
op1    14.747376
op2     0.310016
op3    14.237735
dtype: float64


**Finding:** op1 and op3 have a huge gap either way (~0.002 vs. ~14+). op2 is the tight one: 0.0003 (single-condition) vs. 0.31 (multi-condition). At the originally-chosen threshold of 1.0, op2's multi-condition value (0.31) is actually *below* the threshold — op2 alone wouldn't correctly flag FD002/FD004 as multi-condition at that setting. The check still works today only because `_is_single_condition` requires *all three* columns to agree (`.all()`), and op1/op3 unambiguously do.

**Decision:** lower `SINGLE_CONDITION_STD_THRESHOLD` from 1.0 to 0.1. That sits comfortably between op2's two values (0.0003 and 0.31) as well as op1/op3's, so every column independently discriminates correctly — the classification no longer depends on the AND-across-columns logic to compensate for one close column.